### Imports

In [0]:
import sys
from pathlib import Path

DATABRICKS_ROOT = str(Path.cwd().parent)

if DATABRICKS_ROOT not in sys.path:
    sys.path.insert(0, DATABRICKS_ROOT)

from config.datasets import get_dataset_config

from utilities.io_utils import (
    discover_source_files,
    get_unprocessed_source_files,
    read_dataset,
    write_delta_table,
)

### File Selection

In [0]:
dbutils.widgets.text(
    "dataset_name",
    "customers",
    "Dataset Name",
)

dataset_name = (
    dbutils.widgets
    .get("dataset_name")
    .strip()
)

config = get_dataset_config(dataset_name)

In [0]:
available_files = discover_source_files(config)

print(f"Dataset: {dataset_name}")
print(f"Files discovered: {len(available_files)}")

for file in available_files:
    print(file)

In [0]:
new_files = get_unprocessed_source_files(
    spark,
    config,
)

print(f"New files to process: {len(new_files)}")

for file in new_files:
    print(file)

### Data Ingestion

In [0]:
if not new_files:

    print(
        f"No new files found for {dataset_name}. "
        f"Bronze ingestion skipped."
    )

else:

    raw_df = read_dataset(
        spark=spark,
        config=config,
        source_files=new_files,
    )

    rows_read = raw_df.count()

    write_delta_table(
        spark=spark,
        dataframe=raw_df,
        table_name=config["bronze_table"],
    )

    print(
        f"Completed Bronze ingestion for {dataset_name}. "
        f"Files processed: {len(new_files)}. "
        f"Rows ingested: {rows_read:,}"
    )

In [0]:
bronze_df = spark.table(
    config["bronze_table"]
)

print(f"Dataset: {dataset_name}")
print(f"Rows in Bronze: {bronze_df.count():,}")
print(f"Target table: {config['bronze_table']}")

display(bronze_df.limit(10))